# DPN

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
%matplotlib inline
import seaborn as sns

from tqdm import tqdm
from pprint import pprint

import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)      # decimal places for outputs from numpy
pd.set_option("display.precision", 3) # decimal places for outputs from pandas 
pd.options.display.max_rows=15        # max rows for outputs from pandas

In [2]:
from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix

from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid

In [3]:
%load_ext autoreload
%autoreload 2
from module.dataload import DPN_data
from module.eda import EDA
#import vmodels

### Data Loading

In [4]:
D = DPN_data("../dataset/Sudoscan Working File with Stats.xlsx")
D.load(classification="binary")
D.df

,CODE,AGE,DM_DUR,HBA1C,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,...,INSULIN_nan,HPN_Y,PAOD_Y,DSLPDMIA_Y,CKD_Y,GBS_Y,DEC_VS_Y,DEC_PPS_Y,DEC_LTS_Y,DEC_AR_Y
0,1.0,64.0,7.0,15.00,9.0,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
1,2.0,59.0,1.0,5.60,4.0,19.41,52.3,14.21,61.9,49.3,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,64.0,11.0,7.50,5.0,0.00,0.0,0.00,0.0,35.9,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
3,4.0,53.0,10.0,7.60,8.0,7.86,46.7,7.07,42.5,40.4,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,5.0,57.0,5.0,14.40,1.0,4.19,41.9,3.70,38.2,38.5,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,186.0,52.0,4.0,5.77,6.0,21.56,56.7,13.24,51.6,52.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
186,187.0,33.0,1.0,6.40,2.0,20.63,56.6,19.52,52.6,46.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
187,188.0,36.0,1.0,6.18,3.0,11.45,49.2,13.79,40.2,41.8,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
188,189.0,60.0,5.0,12.20,8.0,5.03,37.9,0.00,0.0,36.3,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0


In [5]:
df = D.df
df
data_cols = list(set(df.columns) - set(D.non_data_cols))
print(len(data_cols), data_cols)

41 ['SPSC_R', 'FWAVE_R', 'CMAPANK_L', 'SSA_L', 'FWAVE_L', 'DEC_PPS_Y', 'INSULIN_nan', 'HBA1C', 'DL_R', 'INSULIN_Y', 'DSLPDMIA_Y', 'DEC_AR_Y', 'DM_DUR', 'SSA_R', 'HPN_Y', 'SEX_M', 'AGE', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'DL_L', 'CMAPKNE_R', 'MCV_R', 'CMAPANK_R', 'SUBJ_Y', 'HAND_MEAN_ESC', 'CAS', 'SSC_R', 'CMAPKNE_L', 'SPSA_R', 'DEC_VS_Y', 'PAOD_Y', 'DEC_LTS_Y', 'HAND_PCT_ASYM', 'SPSA_L', 'CKD_Y', 'MCV_L', 'MNSI', 'NS', 'SSC_L', 'GBS_Y', 'SPSC_L']


In [6]:
df.columns

Index(['CODE', 'AGE', 'DM_DUR', 'HBA1C', 'MNSI', 'SSA_L', 'SSC_L', 'SPSA_L',
       'SPSC_L', 'MCV_L', 'DL_L', 'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L', 'SSA_R',
       'SSC_R', 'SPSA_R', 'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R',
       'FWAVE_R', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC',
       'HAND_PCT_ASYM', 'NS', 'CAS', 'Confirmed_Binary_DPN', 'SEX_M', 'SUBJ_Y',
       'INSULIN_Y', 'INSULIN_nan', 'HPN_Y', 'PAOD_Y', 'DSLPDMIA_Y', 'CKD_Y',
       'GBS_Y', 'DEC_VS_Y', 'DEC_PPS_Y', 'DEC_LTS_Y', 'DEC_AR_Y'],
      dtype='object')

In [7]:
# check codes of patients that were dropped
set (range(1, 191)) - set(D.df.CODE.values) 

set()

## Multi-Class Study
('Negative', 'Possible', 'Probable', 'Confirmed')

### Data Inspection

In [8]:
D.multi_classes_labels

['Negative', 'Possible', 'Probable', 'Confirmed']

In [9]:
X = df[data_cols]
y = df['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 41), (190,))

In [10]:
y

0      1
1      0
2      1
3      1
4      1
      ..
185    0
186    0
187    0
188    1
189    1
Name: Confirmed_Binary_DPN, Length: 190, dtype: int32

In [11]:
# get supports for each class
y.sum()

130

### Train Test Split

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)
# y_val
X_train

(142, 41) (48, 41) (142,) (48,)


,SPSC_R,FWAVE_R,CMAPANK_L,SSA_L,FWAVE_L,DEC_PPS_Y,INSULIN_nan,HBA1C,DL_R,INSULIN_Y,...,DEC_LTS_Y,HAND_PCT_ASYM,SPSA_L,CKD_Y,MCV_L,MNSI,NS,SSC_L,GBS_Y,SPSC_L
170,39.0,50.6,2.43,9.27,49.5,1.0,0.0,6.41,3.20,1.0,...,0.0,0.0,4.29,0.0,42.7,7.0,56.0,46.9,0.0,40.2
116,0.0,54.2,4.19,0.00,52.8,1.0,0.0,9.20,3.95,1.0,...,1.0,9.0,0.00,0.0,40.0,6.0,46.0,0.0,0.0,0.0
13,40.2,52.2,13.07,8.97,52.5,1.0,0.0,7.30,3.10,0.0,...,0.0,18.0,6.86,0.0,43.2,2.0,40.0,47.2,0.0,39.0
33,54.9,41.9,7.90,11.20,40.2,1.0,0.0,6.40,3.55,0.0,...,1.0,30.0,14.63,0.0,49.1,6.0,48.0,57.0,0.0,51.8
44,42.7,48.1,10.65,11.57,47.7,0.0,0.0,6.90,4.45,0.0,...,1.0,7.0,8.50,1.0,47.5,3.0,65.0,51.6,0.0,46.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,51.7,44.1,16.76,12.72,43.9,0.0,0.0,5.50,3.25,0.0,...,0.0,0.0,2.70,0.0,50.0,5.0,47.0,54.4,0.0,49.2
39,40.8,54.3,1.53,6.47,54.7,0.0,0.0,7.50,3.90,1.0,...,1.0,7.0,3.52,0.0,36.2,9.0,52.0,44.9,0.0,41.8
128,48.5,46.6,10.29,14.40,46.4,1.0,0.0,4.90,3.45,0.0,...,1.0,0.0,11.03,0.0,45.1,4.0,34.0,47.6,0.0,45.5
133,0.0,62.7,7.06,14.36,61.0,0.0,0.0,6.60,4.80,0.0,...,0.0,14.0,0.00,1.0,32.1,3.0,64.0,40.9,0.0,0.0


### Dummy Classifier

In [13]:

clf = DummyClassifier(strategy="most_frequent").fit(X_val, y_val) 
y_pred = clf.predict(X_val)

In [14]:
y_pred

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1])

In [15]:
cm = confusion_matrix(
    # np.argmax(y_val, axis=1), # convert 2D predictions to 1D
    # np.argmax(y_pred, axis=1) # convert 2D predictions to 1D
    y_val,
    y_pred
)
cm

array([[ 0, 15],
       [ 0, 33]], dtype=int64)

### Random Forest

In [16]:
clf = RandomForestClassifier(max_features = 4, random_state = 0).fit(X_train, y_train)
y_pred = clf.predict(X_val)

cm = confusion_matrix(
    y_val,
    y_pred
)

print(cm)

pprint(
    EDA.binary_classification_metrics(
        cm,
        labels=D.multi_classes_labels, verbosity=2)
)

[[15  0]
 [ 1 32]]
sensitivity: 0.970
specificity: 1.000
youden index: 0.970
accuracy: 0.979
precision: 1.000
f1: 0.985

tpr: 0.000
plr: inf
---
['Negative', 'Possible', 'Probable', 'Confirmed']
TN 15   FP 0
FN 1   TP 32
[[15  0]
 [ 1 32]]

{'accuracy': 0.9791666666666666,
 'f1': 0.9846153846153847,
 'plr': inf,
 'precision': 1.0,
 'sensitivity': 0.9696969696969697,
 'specificity': 1.0,
 'tpr': 0.0,
 'youden_index': 0.9696969696969697}


### Decision Tree

In [17]:
clf = DecisionTreeClassifier(max_depth = 4, min_samples_leaf = 8, random_state = 0).fit(X_train, y_train)
y_pred = clf.predict(X_val)

cm = confusion_matrix(
    y_val,
    y_pred
)

print(cm)

pprint(
    EDA.binary_classification_metrics(
        cm, 
        labels=D.multi_classes_labels, verbosity=2)
)

[[12  3]
 [ 4 29]]
sensitivity: 0.879
specificity: 0.800
youden index: 0.679
accuracy: 0.854
precision: 0.906
f1: 0.892

tpr: 0.200
plr: 4.394
---
['Negative', 'Possible', 'Probable', 'Confirmed']
TN 12   FP 3
FN 4   TP 29
[[12  3]
 [ 4 29]]

{'accuracy': 0.8541666666666666,
 'f1': 0.8923076923076922,
 'plr': 4.393939393939394,
 'precision': 0.90625,
 'sensitivity': 0.8787878787878788,
 'specificity': 0.8,
 'tpr': 0.2,
 'youden_index': 0.6787878787878787}


### XGBoost

In [18]:
import xgboost as xgb

with open('model_configs/xgb/xgb_binary_def_params.json', 'r') as file:
    json_string = json.load(file)

clf = xgb.XGBClassifier(
    **json_string
    ).fit(X_train, y_train)

y_pred = clf.predict(X_val)

cm = confusion_matrix(
    y_val,
    y_pred
)

print(cm)

pprint(
    EDA.binary_classification_metrics(
        cm,
        labels=D.multi_classes_labels, verbosity=2)
)

[[ 9  6]
 [ 1 32]]
sensitivity: 0.970
specificity: 0.600
youden index: 0.570
accuracy: 0.854
precision: 0.842
f1: 0.901

tpr: 0.400
plr: 2.424
---
['Negative', 'Possible', 'Probable', 'Confirmed']
TN 9   FP 6
FN 1   TP 32
[[ 9  6]
 [ 1 32]]

{'accuracy': 0.8541666666666666,
 'f1': 0.9014084507042254,
 'plr': 2.4242424242424243,
 'precision': 0.8421052631578947,
 'sensitivity': 0.9696969696969697,
 'specificity': 0.6,
 'tpr': 0.4,
 'youden_index': 0.5696969696969698}


### LogisticRegression

In [19]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=100).fit(X_train, y_train)

y_pred = clf.predict(X_val)

cm = confusion_matrix(
    y_val,
    y_pred
)

print(cm)

pprint(
    EDA.binary_classification_metrics(
        cm,
        labels=D.multi_classes_labels, verbosity=2)
)

[[11  4]
 [ 3 30]]
sensitivity: 0.909
specificity: 0.733
youden index: 0.642
accuracy: 0.854
precision: 0.882
f1: 0.896

tpr: 0.267
plr: 3.409
---
['Negative', 'Possible', 'Probable', 'Confirmed']
TN 11   FP 4
FN 3   TP 30
[[11  4]
 [ 3 30]]

{'accuracy': 0.8541666666666666,
 'f1': 0.8955223880597014,
 'plr': 3.409090909090909,
 'precision': 0.8823529411764706,
 'sensitivity': 0.9090909090909091,
 'specificity': 0.7333333333333333,
 'tpr': 0.26666666666666666,
 'youden_index': 0.6424242424242423}


### Grid Search Optimization

#### grid_search_nocv

In [20]:
def grid_search_nocv(estimator, param_grid, X, y, test_size,                      
                     random_state, verbosity, labels="", multiclass=True, 
                     scoring='youden_index', score_aggregation='weighted_avg'):
    
    def get_score(metrics):
        # metrics is the return value of multiclass_metrics(), which is a dictionary of dictionaries
        assert scoring in ['sensitivity', '', 'specificity', 'youden_index', 'accuracy'], f'Invalid scoring parameter  {scoring}'
        assert score_aggregation in ['per_class', 'macro_avg', 'weighted_avg'], f'Invalid score aggregation parameter {score_aggregation}'
        if scoring=='accuracy':
            assert score_aggregation=='macro_avg', f'Score aggregation should be macro_avg for accuracy '
        return metrics[score_aggregation][scoring] 

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=random_state)    
    best_score = 0
    best_grid = None
    best_scores = []
    best_grids = []
    for g in tqdm(ParameterGrid(param_grid)):
        estimator.set_params(**g)
        try:
            estimator.fit(X_train,y_train)
            y_pred = estimator.predict(X_val)

            if multiclass:
                y_pred = np.argmax(y_pred, axis=1)
                y_val = np.argmax(y_val, axis=1)                

            cm = confusion_matrix(y_val, y_pred)            
            metrics = EDA.binary_classification_metrics(cm, labels=labels, verbosity=0)
            pprint(metrics)
            score = get_score(metrics)
            

            if verbosity>1:
                print(score, end=" ")
            if score > best_score: # save if best
                best_score = score
                best_grid = g                
                best_scores.append(best_score)
                best_grids.append(best_grid)
                if verbosity>1:
                    print("\n", best_score, best_grid, "\n")
        except:
            continue
    return best_scores, best_grids

In [21]:
estimator= RandomForestClassifier()
model_class = RandomForestClassifier

train_kwargs = {
    'test_size' : 0.25,
    'random_state' : 42
}

# for test parameters for quick run
test_param_grid = { 
    'n_estimators': [5],
    'max_depth': [None, 4],
    'min_samples_leaf': [1],
    'min_samples_split': [2, 4],
    'criterion': ['gini'],
    'max_features': ['sqrt'],
    'bootstrap': [True],
}

# real grid search parameters
opt_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 4, 8, 12, 16, 24, 32],
    'min_samples_leaf': [1, 2, 4, 8, 12, 16],
    'min_samples_split': [2, 4, 8, 16],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False],
}

In [22]:
param_grid = test_param_grid # comment out if running actual
#param_grid = opt_param_grid # comment out if just testing

scoring='youden_index'
score_aggregation='weighted_avg'
best_scores, best_grids = grid_search_nocv(estimator, param_grid, 
                                                   X, y, test_size=train_kwargs['test_size'], 
                                                   scoring='youden_index', score_aggregation=score_aggregation,
                                                   multiclass=True,
                                                   random_state=train_kwargs['random_state'], verbosity=2)

print(f'Best model -----')
print(f'Criteria: {scoring} ({score_aggregation})')
print(f'Score: {best_scores[-1]}')
print(f'Parameters: {best_grids[-1]}')
#vmodels.save_gridsearch_results(estimator, best_scores, best_grids, test_size=train_kwargs['test_size'], random_state=train_kwargs['random_state'])

100%|██████████| 4/4 [00:00<00:00, 176.76it/s]


Best model -----
Criteria: youden_index (weighted_avg)


IndexError: list index out of range

#### grid_search_cv (with Cross Validation)

In [23]:
def get_score_cv(y_true, y_pred, labels="", result_aggregation='by_statistics', verbosity=0):
    cm = confusion_matrix(y_true, y_pred)
    metrics = EDA.binary_classification_metrics(cm, labels=labels, verbosity=verbosity)
    pprint(metrics)

    # metrics is the return value of multiclass_metrics(), which is a dictionary of dictionaries
    assert scoring in ['sensitivity', '', 'specificity', 'youden_index', 'accuracy'], f'Invalid scoring parameter  {scoring}'
    assert score_aggregation in ['per_class', 'macro_avg', 'weighted_avg'], f'Invalid score aggregation parameter {score_aggregation}'
    if scoring=='accuracy':
        assert score_aggregation=='macro_avg', f'Score aggregation should be macro_avg for accuracy '
    return metrics[score_aggregation][scoring] 


def grid_search_cv(estimator, param_grid, X_train, y_train, 
                   verbosity, labels="", multiclass=True, 
                   scoring='youden_index', score_aggregation='weighted_avg', cv_splits=5):
    
    
    # Wrap scorer for GridSearchCV
    scorer = make_scorer(get_score_cv, greater_is_better=True, labels="", result_aggregation='by_statistics', verbosity=0)    
    grid = GridSearchCV(estimator, param_grid, scoring=scorer, cv=cv_splits)
    if multiclass:
        grid.fit(X_train, np.argmax(y_train, axis=1))
    else:
        grid.fit(X_train, y_train)

    if verbosity>0:
        print("Best params:", grid.best_params_)
        print(f"Best {scoring} ({score_aggregation}):", grid.best_score_)

    return grid

In [24]:
estimator= RandomForestClassifier()
model_class = RandomForestClassifier

# for test parameters for quick run
test_param_grid1 = { 
    'n_estimators': [5],
    'max_depth': [None, 4],
    'min_samples_leaf': [1],
    'min_samples_split': [2, 4],
    'criterion': ['gini'],
    'max_features': ['sqrt'],
    'bootstrap': [True],
}

# for test parameters for quick but wider run
test_param_grid2 = {
    'n_estimators': [100],           # Number of trees; default: 100
    'max_depth': [None, 8],          # How deep each tree is; default: None
    'min_samples_leaf': [1],         # Minimum number of samples per leaf; default: 1
    'min_samples_split': [2, 4],     # Minimum number of samples to split a node; default: 2
    'criterion': ['gini'],           # Splitting criterion; default: 'gini'
    'max_features': ['sqrt'],        # Features to consider at each split; 'sqrt'
    'bootstrap': [True, False],      # Whether to sample with replacement; default: True
}
    

# wider grid search parameters
opt_param_grid = {
    'n_estimators': [50, 100, 200, 300],           # Number of trees
    'max_depth': [None, 8, 16, 24, 32],            # How deep each tree is
    'min_samples_leaf': [1, 2, 4, 8],              # Minimum number of samples per leaf
    'min_samples_split': [2, 4, 8, 16],            # Minimum number of samples to split a node
    'criterion': ['gini', 'entropy'],              # Splitting criterion
    'max_features': ['sqrt', 'log2', None],        # Features to consider at each split
    'bootstrap': [True, False],                    # Whether to sample with replacement
}

# uncomment parameter grid to use
# param_grid = test_param_grid1
param_grid = test_param_grid2 
# param_grid = opt_param_grid  

# uncomment scoring and aggregation to use
scoring='youden_index'
score_aggregation='weighted_avg'
# scoring='accuracy'
# score_aggregation='macro_avg'

cv_grid = grid_search_cv(
    estimator, param_grid,  X_train, y_train,
    scoring=scoring, score_aggregation=score_aggregation,
    multiclass=True,
    verbosity=1,
    cv_splits=5)

ValueError: `axis` must be fewer than the number of dimensions (1)

##### best model

In [25]:
cv_grid.best_score_, cv_grid.best_params_

NameError: name 'cv_grid' is not defined

In [26]:
cv_grid.best_estimator_

NameError: name 'cv_grid' is not defined

##### Test set performance

In [27]:
y_pred = cv_grid.best_estimator_.predict(X_val)
print(y_pred)
print(np.argmax(y_val, axis=1))

NameError: name 'cv_grid' is not defined

In [28]:
y_val

165    1
2      1
182    0
21     0
10     1
      ..
140    1
15     1
114    1
135    0
80     0
Name: Confirmed_Binary_DPN, Length: 48, dtype: int32

In [29]:
cm = confusion_matrix(np.argmax(y_val, axis=1), y_pred)
metrics = EDA.binary_classification_metrics(cm, labels=D.multi_classes, verbosity=2)

ValueError: `axis` must be fewer than the number of dimensions (1)

## Binary Classifier Confirmed vs. Non-Confirmed

Confirmed vs. (Probable+Possible+Negative)

In [30]:
D = DPN_data("../dataset/Sudoscan Working File with Stats.xlsx")
D.load()
df = D.df
data_cols = list(set(df.columns) - set(D.non_data_cols))
print(data_cols)

['SPSC_R', 'FWAVE_R', 'CMAPANK_L', 'SSA_L', 'FWAVE_L', 'DEC_PPS_Y', 'INSULIN_nan', 'HBA1C', 'DL_R', 'INSULIN_Y', 'DSLPDMIA_Y', 'DEC_AR_Y', 'DM_DUR', 'SSA_R', 'HPN_Y', 'SEX_M', 'AGE', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'DL_L', 'CMAPKNE_R', 'MCV_R', 'CMAPANK_R', 'SUBJ_Y', 'HAND_MEAN_ESC', 'CAS', 'SSC_R', 'CMAPKNE_L', 'SPSA_R', 'DEC_VS_Y', 'PAOD_Y', 'DEC_LTS_Y', 'HAND_PCT_ASYM', 'SPSA_L', 'CKD_Y', 'MCV_L', 'MNSI', 'NS', 'SSC_L', 'GBS_Y', 'SPSC_L']


In [31]:
df['Confirmed_Binary_DPN'].sum()

130

### Data Inspection

In [32]:
X = df[data_cols]
X

,SPSC_R,FWAVE_R,CMAPANK_L,SSA_L,FWAVE_L,DEC_PPS_Y,INSULIN_nan,HBA1C,DL_R,INSULIN_Y,...,DEC_LTS_Y,HAND_PCT_ASYM,SPSA_L,CKD_Y,MCV_L,MNSI,NS,SSC_L,GBS_Y,SPSC_L
0,0.0,0.0,0.00,0.00,0.0,1.0,0.0,15.00,10.35,1.0,...,1.0,13.0,0.00,0.0,0.0,9.0,42.0,0.0,0.0,0.0
1,61.2,43.3,14.34,19.41,42.5,0.0,0.0,5.60,3.30,0.0,...,0.0,28.0,14.21,0.0,49.3,4.0,50.0,52.3,0.0,61.9
2,0.0,54.7,1.83,0.00,54.4,1.0,0.0,7.50,4.70,1.0,...,1.0,1.0,0.00,0.0,35.9,5.0,50.0,0.0,0.0,0.0
3,42.7,50.9,6.08,7.86,51.0,0.0,0.0,7.60,4.25,1.0,...,0.0,5.0,7.07,0.0,40.4,8.0,57.0,46.7,0.0,42.5
4,39.5,49.9,8.89,4.19,48.3,0.0,0.0,14.40,4.00,1.0,...,1.0,0.0,3.70,0.0,38.5,1.0,54.0,41.9,0.0,38.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,51.6,40.2,16.92,21.56,39.3,1.0,0.0,5.77,3.35,0.0,...,1.0,5.0,13.24,0.0,52.0,6.0,61.0,56.7,0.0,51.6
186,58.1,47.1,11.53,20.63,46.3,0.0,0.0,6.40,3.90,0.0,...,0.0,12.0,19.52,0.0,46.6,2.0,82.0,56.6,0.0,52.6
187,41.2,49.9,11.94,11.45,50.3,1.0,0.0,6.18,3.70,1.0,...,0.0,8.0,13.79,1.0,41.8,3.0,83.0,49.2,0.0,40.2
188,0.0,53.5,5.05,5.03,53.1,1.0,0.0,12.20,4.20,1.0,...,1.0,9.0,0.00,0.0,36.3,8.0,46.0,37.9,0.0,0.0


In [33]:
y = df['Confirmed_Binary_DPN']
y

0      1
1      0
2      1
3      1
4      1
      ..
185    0
186    0
187    0
188    1
189    1
Name: Confirmed_Binary_DPN, Length: 190, dtype: int32

In [34]:
X.shape, y.shape

((190, 41), (190,))

### Train Test Split

In [35]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)
y_val

(142, 41) (48, 41) (142,) (48,)


165    1
2      1
182    0
21     0
10     1
      ..
140    1
15     1
114    1
135    0
80     0
Name: Confirmed_Binary_DPN, Length: 48, dtype: int32

### Random Forest

In [36]:
clf = RandomForestClassifier(max_features = 4, random_state = 0).fit(X_train, y_train)
y_pred = clf.predict(X_val)

### grid_search_cv (with Cross Validation)

In [37]:
def grid_search_cv_binary(estimator, param_grid, X_train, y_train, 
                          verbosity, labels="", scoring='youden_index', cv_splits=5):

    def get_binary_score_cv(y_true, y_pred, labels="", verbosity=0):
        cm = confusion_matrix(y_true, y_pred)
        metrics = EDA.binary_classification_metrics(cm, labels=labels, verbosity=verbosity)
        pprint(metrics)
        return metrics[scoring] 

    # Wrap scorer for GridSearchCV
    scorer = make_scorer(get_binary_score_cv, greater_is_better=True, labels="", verbosity=0)    
    grid = GridSearchCV(estimator, param_grid, scoring=scorer, cv=cv_splits)
    grid.fit(X_train, y_train)

    if verbosity>0:
        print("Best params:", grid.best_params_)
        print(f"Best {scoring} ({score_aggregation}):", grid.best_score_)
    return grid

In [44]:
with open('model_configs/xgb/xgb_binary_param_grid.json', 'r') as file:
    json_string = json.load(file)

In [45]:
# for test parameters for quick but wider run
test_param_grid2 = {
    'n_estimators': [100],           # Number of trees; default: 100
    'max_depth': [None, 8],          # How deep each tree is; default: None
    'min_samples_leaf': [1],         # Minimum number of samples per leaf; default: 1
    'min_samples_split': [2, 4],     # Minimum number of samples to split a node; default: 2
    'criterion': ['gini'],           # Splitting criterion; default: 'gini'
    'max_features': ['sqrt'],        # Features to consider at each split; 'sqrt'
    'bootstrap': [True, False],      # Whether to sample with replacement; default: True
}
    

# wider grid search parameters
opt_param_grid = {
    'n_estimators': [50, 100, 200, 300],           # Number of trees
    'max_depth': [None, 8, 16, 24, 32],            # How deep each tree is
    'min_samples_leaf': [1, 2, 4, 8],              # Minimum number of samples per leaf
    'min_samples_split': [2, 4, 8, 16],            # Minimum number of samples to split a node
    'criterion': ['gini', 'entropy'],              # Splitting criterion
    'max_features': ['sqrt', 'log2', None],        # Features to consider at each split
    'bootstrap': [True, False],                    # Whether to sample with replacement
}

# uncomment parameter grid to use
param_grid = json_string 
# param_grid = opt_param_grid  

# uncomment scoring to use
scoring='youden_index'

binary_cv_grid = grid_search_cv_binary(
    estimator, param_grid,  X_train, y_train,
    scoring=scoring, 
    labels=["Confirmed", "Non-confirmed"],
    verbosity=1,
    cv_splits=5)

MemoryError: 

##### best model

In [ ]:
binary_cv_grid.best_score_, binary_cv_grid.best_params_

##### Test set performance

In [ ]:
y_pred = binary_cv_grid.best_estimator_.predict(X_val)
print(y_pred)
print(y_val)

In [ ]:
EDA.binary_classification_metrics(
    confusion_matrix(y_val, y_pred), 
    labels=["Confirmed", "Non-confirmed"], 
    verbosity=2)

In [ ]:
X

### Correlation Analysis

In [ ]:
D = DPN_data("SudoscanRaw.xlsx")
D.load(one_hot_encode=False)
df = D.df
df.replace('Y', 1, inplace=True)
df.replace('N', 0, inplace=True)
df.replace('M', 1, inplace=True)
df.replace('F', 0, inplace=True)
df.head()

In [ ]:
def plot_heatmap(corr_matrix, figsize=(6, 4)):
    # Plot heatmap
    plt.figure(figsize=figsize)
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title('Correlation Matrix Heatmap')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_heatmap_thresholded(corr_matrix, threshold=0.9, figsize=(6, 4)):
    mask = corr_matrix.abs() < threshold
    mask |= np.eye(len(corr_matrix)).astype(bool)  # keep diagonal clear

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap="coolwarm", center=0)
    plt.title("Correlation Matrix (Only > 0.8 Shown)")
    plt.show()

In [ ]:
plot_heatmap(df[D.sudo_cols].corr(), figsize=(12, 8))

In [ ]:
plot_heatmap(df[D.ncs_cols].corr(), figsize=(12, 8))

In [ ]:
plot_heatmap_thresholded(df[D.ncs_cols].corr(), threshold=0.8, figsize=(12, 8))

In [ ]:
binary_cols = ["AGE", 'DM_DUR', 'HBA1C', 'MNSI'] + D.commorbidity_cols + D.neuro_cols
plot_heatmap(df[binary_cols].corr(), figsize=(12, 8))

In [ ]:
X.columns